
# 🚢 Titanic Dataset — Feature Engineering

# GTU accoding

**Aim:** Handle missing values, encode categorical variables, scale numeric features, and create new features so the dataset is ready for ML models.


In [1]:
# Cell 1 — Imports + Load dataset
import pandas as pd
import numpy as np
from sklearn.datasets import fetch_openml
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
import warnings
warnings.filterwarnings("ignore")

# Load Titanic dataset from OpenML (as DataFrame)
titanic = fetch_openml("titanic", version=1, as_frame=True)
df_raw = titanic.frame.copy()   # keep raw untouched copy
df = df_raw.copy()              # we will modify 'df'

# Quick overview
print("Columns:", list(df.columns))
print("\nShape:", df.shape)
print("\nFirst 10 rows (raw):")
display(df.head(10))


Columns: ['pclass', 'survived', 'name', 'sex', 'age', 'sibsp', 'parch', 'ticket', 'fare', 'cabin', 'embarked', 'boat', 'body', 'home.dest']

Shape: (1309, 14)

First 10 rows (raw):


,pclass,survived,name,sex,age,sibsp,parch,ticket,fare,cabin,embarked,boat,body,home.dest
0,1,1,"Allen, Miss. Elisabeth Walton",female,29.0000,0,0,24160,211.3375,B5,S,2,NaN,"St Louis, MO"
1,1,1,"Allison, Master. Hudson Trevor",male,0.9167,1,2,113781,151.5500,C22 C26,S,11,NaN,"Montreal, PQ / Chesterville, ON"
2,1,0,"Allison, Miss. Helen Loraine",female,2.0000,1,2,113781,151.5500,C22 C26,S,NaN,NaN,"Montreal, PQ / Chesterville, ON"
3,1,0,"Allison, Mr. Hudson Joshua Creighton",male,30.0000,1,2,113781,151.5500,C22 C26,S,NaN,135.0,"Montreal, PQ / Chesterville, ON"
4,1,0,"Allison, Mrs. Hudson J C (Bessie Waldo Daniels)",female,25.0000,1,2,113781,151.5500,C22 C26,S,NaN,NaN,"Montreal, PQ / Chesterville, ON"
5,1,1,"Anderson, Mr. Harry",male,48.0000,0,0,19952,26.5500,E12,S,3,NaN,"New York, NY"
6,1,1,"Andrews, Miss. Kornelia Theodosia",female,63.0000,1,0,13502,77.9583,D7,S,10,NaN,"Hudson, NY"
7,1,0,"Andrews, Mr. Thomas Jr",male,39.0000,0,0,112050,0.0000,A36,S,NaN,NaN,"Belfast, NI"
8,1,1,"Appleton, Mrs. Edward Dale (Charlotte Lamson)",female,53.0000,2,0,11769,51.4792,C101,S,D,NaN,"Bayside, Queens, NY"
9,1,0,"Artagaveytia, Mr. Ramon",male,71.0000,0,0,PC 17609,49.5042,NaN,C,NaN,22.0,"Montevideo, Uruguay"


## 🔍 Step 1 — Initial exploration of missing values
*We will inspect how many missing values each column has (counts and percent).*


In [2]:
# Cell 2 — Missing values summary
missing_count = df.isnull().sum()
missing_percent = (missing_count / len(df)) * 100
missing_df = pd.DataFrame({
    "missing_count": missing_count,
    "missing_percent": missing_percent
}).sort_values("missing_percent", ascending=False)

display(missing_df)


,missing_count,missing_percent
body,1188,90.756303
cabin,1014,77.463713
boat,823,62.872422
home.dest,564,43.086325
age,263,20.091673
embarked,2,0.152788
fare,1,0.076394
pclass,0,0.000000
survived,0,0.000000
name,0,0.000000


## ✂️ Step 2 — Decide and drop columns with many missing values
Common practice: drop columns with too many missing values (e.g., `cabin`, `boat`, `body`, `home.dest`).
We will detect which columns exceed a chosen threshold (here 40%) and drop them.


In [3]:
# Cell 3 — Auto-detect columns to drop (>40% missing)
threshold_pct = 40.0
cols_to_drop = missing_df[missing_df["missing_percent"] > threshold_pct].index.tolist()

print("Columns with >{}% missing (will be dropped):\n".format(threshold_pct), cols_to_drop)

# But if you specifically want to drop the syllabus columns use:
specific_drop = ["cabin","boat","body","home.dest"]
# Use the union of both sets (so we don't accidentally miss intended syllabus drops)
cols_to_drop = list(set(cols_to_drop) | set(specific_drop))

# Drop safely (errors="ignore" if not present)
original_cols = df.columns.tolist()
df = df.drop(columns=cols_to_drop, errors="ignore")

removed_cols = set(original_cols) - set(df.columns)
print("\nRemoved columns:", removed_cols)
print("\nShape after drop:", df.shape)


Columns with >40.0% missing (will be dropped):
 ['body', 'cabin', 'boat', 'home.dest']

Removed columns: {'body', 'boat', 'cabin', 'home.dest'}

Shape after drop: (1309, 10)


## 🩺 Step 3 — Show BEFORE imputation for important columns
We show the first 10 rows and missing counts for `age`, `fare`, and `embarked` (the columns we will impute).


In [4]:
# Cell 4 — Before imputation (sample + missing counts)
cols_check = [c for c in ["age", "fare", "embarked"] if c in df.columns]
print("Columns to be imputed:", cols_check)
print("\nMissing before imputation:\n", df[cols_check].isnull().sum())
display(df[cols_check].head(10))


Columns to be imputed: ['age', 'fare', 'embarked']

Missing before imputation:
 age         263
fare          1
embarked      2
dtype: int64


,age,fare,embarked
0,29.0000,211.3375,S
1,0.9167,151.5500,S
2,2.0000,151.5500,S
3,30.0000,151.5500,S
4,25.0000,151.5500,S
5,48.0000,26.5500,S
6,63.0000,77.9583,S
7,39.0000,0.0000,S
8,53.0000,51.4792,S
9,71.0000,49.5042,C


## 🧩 Step 4 — Impute missing values
- `age` and `fare` → median (robust to outliers)  
- `embarked` → most frequent (mode)


In [5]:
# Cell 5 — Imputation
imputer_median = SimpleImputer(strategy="median")
imputer_mode = SimpleImputer(strategy="most_frequent")

# numeric impute (if columns exist)
for col in ["age", "fare"]:
    if col in df.columns:
        df[col] = imputer_median.fit_transform(df[[col]]).ravel()   # flatten 2D -> 1D

# categorical impute
if "embarked" in df.columns:
    df["embarked"] = imputer_mode.fit_transform(df[["embarked"]]).ravel()

# Show result
print("Missing values AFTER imputation:\n")
print(df.isnull().sum().head(10))
display(df.head(10))


Missing values AFTER imputation:

pclass      0
survived    0
name        0
sex         0
age         0
sibsp       0
parch       0
ticket      0
fare        0
embarked    0
dtype: int64


,pclass,survived,name,sex,age,sibsp,parch,ticket,fare,embarked
0,1,1,"Allen, Miss. Elisabeth Walton",female,29.0000,0,0,24160,211.3375,S
1,1,1,"Allison, Master. Hudson Trevor",male,0.9167,1,2,113781,151.5500,S
2,1,0,"Allison, Miss. Helen Loraine",female,2.0000,1,2,113781,151.5500,S
3,1,0,"Allison, Mr. Hudson Joshua Creighton",male,30.0000,1,2,113781,151.5500,S
4,1,0,"Allison, Mrs. Hudson J C (Bessie Waldo Daniels)",female,25.0000,1,2,113781,151.5500,S
5,1,1,"Anderson, Mr. Harry",male,48.0000,0,0,19952,26.5500,S
6,1,1,"Andrews, Miss. Kornelia Theodosia",female,63.0000,1,0,13502,77.9583,S
7,1,0,"Andrews, Mr. Thomas Jr",male,39.0000,0,0,112050,0.0000,S
8,1,1,"Appleton, Mrs. Edward Dale (Charlotte Lamson)",female,53.0000,2,0,11769,51.4792,S
9,1,0,"Artagaveytia, Mr. Ramon",male,71.0000,0,0,PC 17609,49.5042,C


## ✅ Observation after imputation
You should see `age`, `fare`, and `embarked` with **0 missing** now (or reduced).


## 🔁 Step 5 — Encoding categorical variables (Before/After)
We'll convert categorical columns (sex, embarked, pclass) to numeric dummy variables.
If these columns are absent (maybe because you already encoded earlier), we'll detect suitable categorical columns automatically.


In [6]:
# Cell 6 — Detect categorical columns to encode
# Preferred list
preferred = ["sex", "embarked", "pclass"]

# Determine which of preferred actually exist
to_encode = [c for c in preferred if c in df.columns]

# If none of preferred exist, fallback: select object-type columns except 'name' and 'ticket' (we might handle 'name' separately)
if not to_encode:
    fallback = [c for c in df.select_dtypes(include="object").columns if c not in ("name","ticket")]
    to_encode = fallback

print("Columns to encode (one-hot):", to_encode)

# Show BEFORE encoding sample (if available)
if to_encode:
    print("\nSample BEFORE encoding:")
    display(df[to_encode].head(8))

# Perform one-hot encoding safely (only on listed columns)
df = pd.get_dummies(df, columns=to_encode, drop_first=True)

# Show AFTER encoding columns (list a few)
print("\nColumns after encoding (some):")
print(df.columns.tolist()[:30])
display(df.head(8))


Columns to encode (one-hot): ['sex', 'embarked', 'pclass']

Sample BEFORE encoding:


,sex,embarked,pclass
0,female,S,1
1,male,S,1
2,female,S,1
3,male,S,1
4,female,S,1
5,male,S,1
6,female,S,1
7,male,S,1



Columns after encoding (some):
['survived', 'name', 'age', 'sibsp', 'parch', 'ticket', 'fare', 'sex_male', 'embarked_Q', 'embarked_S', 'pclass_2', 'pclass_3']


,survived,name,age,sibsp,parch,ticket,fare,sex_male,embarked_Q,embarked_S,pclass_2,pclass_3
0,1,"Allen, Miss. Elisabeth Walton",29.0000,0,0,24160,211.3375,False,False,True,False,False
1,1,"Allison, Master. Hudson Trevor",0.9167,1,2,113781,151.5500,True,False,True,False,False
2,0,"Allison, Miss. Helen Loraine",2.0000,1,2,113781,151.5500,False,False,True,False,False
3,0,"Allison, Mr. Hudson Joshua Creighton",30.0000,1,2,113781,151.5500,True,False,True,False,False
4,0,"Allison, Mrs. Hudson J C (Bessie Waldo Daniels)",25.0000,1,2,113781,151.5500,False,False,True,False,False
5,1,"Anderson, Mr. Harry",48.0000,0,0,19952,26.5500,True,False,True,False,False
6,1,"Andrews, Miss. Kornelia Theodosia",63.0000,1,0,13502,77.9583,False,False,True,False,False
7,0,"Andrews, Mr. Thomas Jr",39.0000,0,0,112050,0.0000,True,False,True,False,False


## 🔎 Step 6 — Scaling numeric features (Before/After)
We'll scale `age` and `fare` using `StandardScaler` (mean=0, std=1).  
We will show the raw values next to scaled values for comparison.


In [7]:
# Cell 7 — Scaling
scaler = StandardScaler()
scale_cols = [c for c in ["age", "fare"] if c in df.columns]

# Save before values for display
before_scale = df[scale_cols].head(10).copy()

# Apply scaling
if scale_cols:
    df[scale_cols] = scaler.fit_transform(df[scale_cols])

# Show before and after (first 10 rows)
if scale_cols:
    after_scale = df[scale_cols].head(10).copy().reset_index(drop=True)
    before_scale = before_scale.reset_index(drop=True)
    compare = pd.concat([before_scale.add_suffix("_before"), after_scale.add_suffix("_scaled")], axis=1)
    display(compare)
else:
    print("No numeric columns found to scale.")


,age_before,fare_before,age_scaled,fare_scaled
0,29.0000,211.3375,-0.039005,3.442584
1,0.9167,151.5500,-2.215952,2.286639
2,2.0000,151.5500,-2.131977,2.286639
3,30.0000,151.5500,0.038512,2.286639
4,25.0000,151.5500,-0.349075,2.286639
5,48.0000,26.5500,1.433827,-0.130140
6,63.0000,77.9583,2.596589,0.863800
7,39.0000,0.0000,0.736169,-0.643464
8,53.0000,51.4792,1.821414,0.351847
9,71.0000,49.5042,3.216729,0.313661


## ➕ Step 7 — Create new features (FamilySize, IsAlone, Title)
- `family_size = sibsp + parch + 1`  
- `is_alone = 1 if family_size == 1 else 0`  
- `title` from `name` (Mr, Mrs, Miss, Master, Officer, etc.) and then dummy-encode the title.


In [8]:
# Cell 8 — New features
# Family size & is_alone
if all(c in df.columns for c in ["sibsp", "parch"]):
    df["family_size"] = df["sibsp"] + df["parch"] + 1
    df["is_alone"] = (df["family_size"] == 1).astype(int)
    print("Added: family_size, is_alone")

# Title extraction (if 'name' present)
if "name" in df.columns:
    df["title"] = df["name"].str.extract(r',\s*([^\.]+)\.', expand=False).str.strip()
    # Map uncommon titles into groups
    df["title"] = df["title"].replace({
        "Mlle":"Miss", "Ms":"Miss", "Mme":"Mrs",
        "Countess":"Lady", "Lady":"Lady", "Dona":"Lady",
        "Don":"Sir", "Sir":"Sir", "Jonkheer":"Sir",
        "Col":"Officer", "Major":"Officer", "Capt":"Officer", "Dr":"Officer"
    }).fillna("Unknown")
    # Show title counts
    print("\nTitle value counts:")
    display(df["title"].value_counts().head(12))
    # Create dummies
    df = pd.get_dummies(df, columns=["title"], drop_first=True)

display(df[["family_size", "is_alone"]].head(8) if "family_size" in df.columns else "family features not created")


Added: family_size, is_alone

Title value counts:


title
Mr              757
Miss            264
Mrs             198
Master           61
Officer          15
Rev               8
Sir               3
Lady              2
the Countess      1
Name: count, dtype: int64

,family_size,is_alone
0,1,1
1,4,0
2,4,0
3,4,0
4,4,0
5,1,1
6,2,0
7,1,1


## 🧾 Step 8 — Final check and save
Show final shape, column list, and a head sample. Save to CSV for submission/use.


In [9]:
# Cell 9 — Final dataset overview
print("Final shape:", df.shape)
print("\nSample columns (first 50):")
print(df.columns.tolist()[:50])

print("\nFirst 10 rows of engineered dataset:")
display(df.head(10))

# Save final engineered dataset (optional)
df.to_csv("titanic_featured_engineered.csv", index=False)
print("\nSaved: titanic_featured_engineered.csv")


Final shape: (1309, 22)

Sample columns (first 50):
['survived', 'name', 'age', 'sibsp', 'parch', 'ticket', 'fare', 'sex_male', 'embarked_Q', 'embarked_S', 'pclass_2', 'pclass_3', 'family_size', 'is_alone', 'title_Master', 'title_Miss', 'title_Mr', 'title_Mrs', 'title_Officer', 'title_Rev', 'title_Sir', 'title_the Countess']

First 10 rows of engineered dataset:


,survived,name,age,sibsp,parch,ticket,fare,sex_male,embarked_Q,embarked_S,...,family_size,is_alone,title_Master,title_Miss,title_Mr,title_Mrs,title_Officer,title_Rev,title_Sir,title_the Countess
0,1,"Allen, Miss. Elisabeth Walton",-0.039005,0,0,24160,3.442584,False,False,True,...,1,1,False,True,False,False,False,False,False,False
1,1,"Allison, Master. Hudson Trevor",-2.215952,1,2,113781,2.286639,True,False,True,...,4,0,True,False,False,False,False,False,False,False
2,0,"Allison, Miss. Helen Loraine",-2.131977,1,2,113781,2.286639,False,False,True,...,4,0,False,True,False,False,False,False,False,False
3,0,"Allison, Mr. Hudson Joshua Creighton",0.038512,1,2,113781,2.286639,True,False,True,...,4,0,False,False,True,False,False,False,False,False
4,0,"Allison, Mrs. Hudson J C (Bessie Waldo Daniels)",-0.349075,1,2,113781,2.286639,False,False,True,...,4,0,False,False,False,True,False,False,False,False
5,1,"Anderson, Mr. Harry",1.433827,0,0,19952,-0.130140,True,False,True,...,1,1,False,False,True,False,False,False,False,False
6,1,"Andrews, Miss. Kornelia Theodosia",2.596589,1,0,13502,0.863800,False,False,True,...,2,0,False,True,False,False,False,False,False,False
7,0,"Andrews, Mr. Thomas Jr",0.736169,0,0,112050,-0.643464,True,False,True,...,1,1,False,False,True,False,False,False,False,False
8,1,"Appleton, Mrs. Edward Dale (Charlotte Lamson)",1.821414,2,0,11769,0.351847,False,False,True,...,3,0,False,False,False,True,False,False,False,False
9,0,"Artagaveytia, Mr. Ramon",3.216729,0,0,PC 17609,0.313661,True,False,False,...,1,1,False,False,True,False,False,False,False,False



Saved: titanic_featured_engineered.csv


## ✅ Conclusion (short)
- Dropped columns with many missing values.  
- Imputed `age` & `fare` with median; `embarked` with mode.  
- Encoded categorical columns using one-hot encoding.  
- Scaled `age` and `fare`.  
- Created new features: `family_size`, `is_alone`, and `title`.  
- Final dataset saved as `titanic_featured_engineered.csv`.
